<a href="https://colab.research.google.com/github/Muqadasnaz12/CP493_FL-IIoT-Intrusion-Detection/blob/main/FL_PAPER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required packages
!pip install kagglehub
!pip install flwr

In [2]:
# Import Libraries

import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")

In [3]:
# Download CIC-IIoT-2025 Dataset

import kagglehub

path = kagglehub.dataset_download(
    "muhammadirfangull/cic-iiot-2025"
)

print("Dataset downloaded to:")
print(path)

Resuming download from 0 bytes (549870975 bytes left)...
Resuming download to /root/.cache/kagglehub/datasets/muhammadirfangull/cic-iiot-2025/1.archive (0/549870975) bytes left.


100%|██████████| 524M/524M [00:32<00:00, 16.8MB/s]

Extracting files...


Dataset downloaded to:
/root/.cache/kagglehub/datasets/muhammadirfangull/cic-iiot-2025/versions/1


In [4]:
# Display dataset files

files = glob.glob(os.path.join(path, "*.csv"))
print("CSV Files Found:")

for file in files:
    print(file)

CSV Files Found:
/root/.cache/kagglehub/datasets/muhammadirfangull/cic-iiot-2025/versions/1/combined_dataset.csv


In [5]:
# load all CSV files

df = pd.concat(
    [pd.read_csv(file) for file in files],
    ignore_index=True
)

print(df.shape)
df.head()

(685671, 94)


,device_name,device_mac,label_full,label1,label2,label3,label4,timestamp,timestamp_start,timestamp_end,...,network_time-delta_min,network_time-delta_std_deviation,network_ttl_avg,network_ttl_max,network_ttl_min,network_ttl_std_deviation,network_window-size_avg,network_window-size_max,network_window-size_min,network_window-size_std_deviation
0,edge1,dc:a6:32:dc:27:d4,attack_ddos_syn-flood-port-80_edge1,attack,ddos,syn-flood-port-80,ddos_syn-flood-port-80,2025-01-23T15:31:10.709000Z_2025-01-23T15:31:2...,2025-01-23T15:31:10.709000Z,2025-01-23T15:31:20.709000Z,...,2.600000e-08,0.000042,64.0,64.0,64.0,0.0,17587.532313,64240.0,0.0,28377.701703
1,edge1,dc:a6:32:dc:27:d4,attack_ddos_syn-flood-port-80_edge1,attack,ddos,syn-flood-port-80,ddos_syn-flood-port-80,2025-01-23T15:31:15.709000Z_2025-01-23T15:31:2...,2025-01-23T15:31:15.709000Z,2025-01-23T15:31:25.709000Z,...,2.500000e-08,0.000041,64.0,64.0,64.0,0.0,17259.307890,64240.0,0.0,28201.477525
2,edge1,dc:a6:32:dc:27:d4,attack_ddos_syn-flood-port-80_edge1,attack,ddos,syn-flood-port-80,ddos_syn-flood-port-80,2025-01-23T15:31:20.709000Z_2025-01-23T15:31:3...,2025-01-23T15:31:20.709000Z,2025-01-23T15:31:30.709000Z,...,2.500000e-08,0.000041,64.0,64.0,64.0,0.0,17185.878307,64240.0,0.0,28161.438527
3,edge1,dc:a6:32:dc:27:d4,attack_ddos_syn-flood-port-80_edge1,attack,ddos,syn-flood-port-80,ddos_syn-flood-port-80,2025-01-23T15:31:25.709000Z_2025-01-23T15:31:3...,2025-01-23T15:31:25.709000Z,2025-01-23T15:31:35.709000Z,...,2.500000e-08,0.000041,64.0,64.0,64.0,0.0,16750.718465,64240.0,0.0,27918.422488
4,edge1,dc:a6:32:dc:27:d4,attack_ddos_syn-flood-port-80_edge1,attack,ddos,syn-flood-port-80,ddos_syn-flood-port-80,2025-01-23T15:31:30.709000Z_2025-01-23T15:31:4...,2025-01-23T15:31:30.709000Z,2025-01-23T15:31:40.709000Z,...,2.600000e-08,0.000041,64.0,64.0,64.0,0.0,16673.446073,64240.0,0.0,27874.486393


In [6]:
print(df.info())
print()
print(df.describe())
print()
print(df.columns.tolist())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 685671 entries, 0 to 685670
Data columns (total 94 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   device_name                           685671 non-null  object 
 1   device_mac                            685671 non-null  object 
 2   label_full                            685671 non-null  object 
 3   label1                                685671 non-null  object 
 4   label2                                685671 non-null  object 
 5   label3                                685671 non-null  object 
 6   label4                                685671 non-null  object 
 7   timestamp                             685671 non-null  object 
 8   timestamp_start                       685671 non-null  object 
 9   timestamp_end                         685671 non-null  object 
 10  log_data-ranges_avg                   685671 non-null  float64
 11  

In [7]:
print(type(df))
print(df.shape)

<class 'pandas.core.frame.DataFrame'>
(685671, 94)


In [8]:
df = df.drop_duplicates()

print(df.shape)

(685671, 94)


In [9]:
columns_to_drop = [
    "device_name",
    "device_mac",
    "timestamp",
    "timestamp_start",
    "timestamp_end",
    "label_full",
    "label2",
    "label3",
    "label4"
]

df = df.drop(columns=columns_to_drop)
print(df.shape)

(685671, 85)


In [10]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = encoder.fit_transform(df[col])

print("Encoding complete.")

Encoding complete.


In [12]:
X = df.drop(columns=["label1"])
y = df["label1"]

print(X.shape)
print(y.shape)

(685671, 84)
(685671,)


In [13]:
#splitting the Data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(548536, 84)
(137135, 84)


In [14]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Model training complete!")

Model training complete!


In [15]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

Accuracy : 0.9737193276698144
Precision: 0.9586154823399691
Recall   : 0.9981156797903538
F1 Score : 0.9779668891986404


In [16]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.94      0.97     57000
           1       0.96      1.00      0.98     80135

    accuracy                           0.97    137135
   macro avg       0.98      0.97      0.97    137135
weighted avg       0.97      0.97      0.97    137135



In [18]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance.head(20))

                              Feature  Importance
43          network_packets_all_count    0.118866
44          network_packets_dst_count    0.106199
41            network_packet-size_min    0.078712
55            network_ports_src_count    0.074976
74             network_time-delta_min    0.068084
51            network_ports_all_count    0.057186
53            network_ports_dst_count    0.047113
72             network_time-delta_avg    0.037811
65        network_tcp-flags-rst_count    0.032906
45          network_packets_src_count    0.022544
83  network_window-size_std_deviation    0.019545
36                    network_mss_max    0.019138
35                    network_mss_avg    0.017232
73             network_time-delta_max    0.016566
37                    network_mss_min    0.016101
27                    network_ips_src    0.014555
75   network_time-delta_std_deviation    0.012277
32             network_macs_dst_count    0.011497
66        network_tcp-flags-syn_count    0.011313


In [19]:
print(type(X_train))
print(X_train.shape)

<class 'pandas.core.frame.DataFrame'>
(548536, 84)


In [20]:
print(type(y_train))
print(y_train.shape)

<class 'pandas.core.series.Series'>
(548536,)


In [21]:
print(type(X_test))
print(type(y_test))

<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.series.Series'>


In [22]:

# FEDERATED LEARNING: DATA PREPARATION
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Standardize the features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert labels to NumPy arrays
y_train_array = y_train.to_numpy().astype(np.float32)
y_test_array = y_test.to_numpy().astype(np.float32)

# convert features to float32 to reduce memory usage
X_train_scaled = X_train_scaled.astype(np.float32)
X_test_scaled = X_test_scaled.astype(np.float32)

print("Training features:", X_train_scaled.shape)
print("Training labels:", y_train_array.shape)
print("Testing features:", X_test_scaled.shape)
print("Testing labels:", y_test_array.shape)

print("\nTraining label distribution:")
print(pd.Series(y_train_array).value_counts())

Training features: (548536, 84)
Training labels: (548536,)
Testing features: (137135, 84)
Testing labels: (137135,)

Training label distribution:
1.0    320537
0.0    227999
Name: count, dtype: int64


In [23]:
# FEDERATED LEARNING: CREATE CLIENT DATASETS

from sklearn.model_selection import StratifiedKFold

NUM_CLIENTS = 5

client_data = []

skf = StratifiedKFold(
    n_splits=NUM_CLIENTS,
    shuffle=True,
    random_state=42
)

for client_id, (_, client_indices) in enumerate(
    skf.split(X_train_scaled, y_train_array)
):
    X_client = X_train_scaled[client_indices]
    y_client = y_train_array[client_indices]
    client_data.append((X_client, y_client))
    unique, counts = np.unique(y_client, return_counts=True)
    distribution = dict(zip(unique.astype(int), counts))

    print(
        f"Client {client_id + 1}: "
        f"{X_client.shape[0]} samples, "
        f"class distribution = {distribution}"
    )

Client 1: 109708 samples, class distribution = {np.int64(0): np.int64(45600), np.int64(1): np.int64(64108)}
Client 2: 109707 samples, class distribution = {np.int64(0): np.int64(45599), np.int64(1): np.int64(64108)}
Client 3: 109707 samples, class distribution = {np.int64(0): np.int64(45600), np.int64(1): np.int64(64107)}
Client 4: 109707 samples, class distribution = {np.int64(0): np.int64(45600), np.int64(1): np.int64(64107)}
Client 5: 109707 samples, class distribution = {np.int64(0): np.int64(45600), np.int64(1): np.int64(64107)}


In [24]:
# FEDERATED LEARNING: MODEL DEFINITION

def create_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
        tf.keras.layers.Dense(
            128,
            activation="relu"
        ),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(
            1,
            activation="sigmoid"
        )
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


global_model = create_model()
global_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        10,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,201 (75.00 KB)

 Trainable params: 19,201 (75.00 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
# FEDERATED LEARNING: FEDAVG TRAINING

NUM_ROUNDS = 5
LOCAL_EPOCHS = 1
BATCH_SIZE = 256

round_results = []

for round_num in range(NUM_ROUNDS):
    print(f"\n========== Federated Round {round_num + 1}/{NUM_ROUNDS} ==========")

    global_weights = global_model.get_weights()
    client_weights = []
    client_sizes = []

    for client_id, (X_client, y_client) in enumerate(client_data):
        print(f"\nTraining Client {client_id + 1}...")

        local_model = create_model()
        local_model.set_weights(global_weights)

        local_model.fit(
            X_client,
            y_client,
            epochs=LOCAL_EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=1
        )
        client_weights.append(local_model.get_weights())
        client_sizes.append(len(X_client))
        del local_model

    # Weighted fedavg aggregation
    total_samples = sum(client_sizes)
    averaged_weights = []

    for layer_weights in zip(*client_weights):
        weighted_layer = np.zeros_like(layer_weights[0])
        for client_weight, client_size in zip(layer_weights, client_sizes):
            weighted_layer += client_weight * (client_size / total_samples)
        averaged_weights.append(weighted_layer)
    global_model.set_weights(averaged_weights)

    # eval global model after each round
    probabilities = global_model.predict(
        X_test_scaled,
        batch_size=1024,
        verbose=0
    ).ravel()

    predictions = (probabilities >= 0.5).astype(int)

    round_accuracy = accuracy_score(y_test_array, predictions)
    round_precision = precision_score(y_test_array, predictions)
    round_recall = recall_score(y_test_array, predictions)
    round_f1 = f1_score(y_test_array, predictions)

    round_results.append({
        "Round": round_num + 1,
        "Accuracy": round_accuracy,
        "Precision": round_precision,
        "Recall": round_recall,
        "F1 Score": round_f1
    })

    print("\nGlobal model results:")
    print(f"Accuracy : {round_accuracy:.6f}")
    print(f"Precision: {round_precision:.6f}")
    print(f"Recall   : {round_recall:.6f}")
    print(f"F1 Score : {round_f1:.6f}")


========== Federated Round 1/5 ==========

Training Client 1...
429/429 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9062 - loss: 0.2623

Training Client 2...
429/429 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9074 - loss: 0.2603

Training Client 3...
429/429 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.9065 - loss: 0.2625

Training Client 4...
429/429 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9049 - loss: 0.2640

Training Client 5...
429/429 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9065 - loss: 0.2612

Global model results:
Accuracy : 0.924826
Precision: 0.893430
Recall   : 0.989368
F1 Score : 0.938955

========== Federated Round 2/5 ==========

Training Client 1...
429/429 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9246 - loss: 0.2193

Training Client 2...
429/429 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9256 - loss: 0.2160

Training Client 3...
429/429 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9258 - loss: 0.2160

Training Client 4...
429/429 ━━━━━

In [27]:
# FINAL FEDERATED LEARNING RESULTS

federated_results_df = pd.DataFrame(round_results)

print(federated_results_df)

final_probabilities = global_model.predict(
    X_test_scaled,
    batch_size=1024,
    verbose=0
).ravel()

federated_predictions = (
    final_probabilities >= 0.5
).astype(int)

federated_accuracy = accuracy_score(
    y_test_array,
    federated_predictions
)

federated_precision = precision_score(
    y_test_array,
    federated_predictions
)

federated_recall = recall_score(
    y_test_array,
    federated_predictions
)

federated_f1 = f1_score(
    y_test_array,
    federated_predictions
)

print("\nFinal Federated Learning Results")
print("--------------------------------")
print("Accuracy :", federated_accuracy)
print("Precision:", federated_precision)
print("Recall   :", federated_recall)
print("F1 Score :", federated_f1)

print("\nClassification Report:")
print(
    classification_report(
        y_test_array,
        federated_predictions,
        digits=4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_array,
        federated_predictions
    )
)

   Round  Accuracy  Precision    Recall  F1 Score
0      1  0.924826   0.893430  0.989368  0.938955
1      2  0.932884   0.903664  0.990766  0.945212
2      3  0.936260   0.907826  0.991602  0.947866
3      4  0.939906   0.912899  0.991789  0.950710
4      5  0.943311   0.916860  0.993037  0.953429

Final Federated Learning Results
--------------------------------
Accuracy : 0.9433113355452656
Precision: 0.9168596545804385
Recall   : 0.993036750483559
F1 Score : 0.9534290232914789

Classification Report:
              precision    recall  f1-score   support

         0.0     0.9889    0.8734    0.9276     57000
         1.0     0.9169    0.9930    0.9534     80135

    accuracy                         0.9433    137135
   macro avg     0.9529    0.9332    0.9405    137135
weighted avg     0.9468    0.9433    0.9427    137135


Confusion Matrix:
[[49784  7216]
 [  558 79577]]


In [28]:
# CENTRALIZED VS FEDERATED COMPARISON

comparison_df = pd.DataFrame({
    "Model": [
        "Centralized Random Forest",
        "Federated Neural Network"
    ],
    "Accuracy": [
        0.9737193276698144,
        federated_accuracy
    ],
    "Precision": [
        0.9586154823399691,
        federated_precision
    ],
    "Recall": [
        0.9981156797903538,
        federated_recall
    ],
    "F1 Score": [
        0.9779668891986404,
        federated_f1
    ]
})

print(comparison_df.round(4))

                       Model  Accuracy  Precision  Recall  F1 Score
0  Centralized Random Forest    0.9737     0.9586  0.9981    0.9780
1   Federated Neural Network    0.9433     0.9169  0.9930    0.9534
